# 06. Filter Scenario 가설검증

목표는 defect 전체를 다시 분류하는 것이 아니라, SegFormer mask 안에서 미세스크래치로 강하게 의심되는 component만 제거하는 후처리 filter를 비교하는 것이다.

실제 raw image와 mask 경로를 아래 셀에 넣으면 5개 scenario가 각각 어떤 component를 제거하는지 비교할 수 있다. 경로를 비워두면 synthetic sample로 smoke test를 수행한다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "scratch_postprocess_utils.py").exists():
    matches = list(Path.cwd().glob("**/Scratch_Postprocess/scratch_postprocess_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from scratch_postprocess_utils import *

ensure_dirs()
print("project:", NOTEBOOK_DIR)

## 1. 실제 데이터 경로 입력

`USER_IMAGE_PATH`, `USER_MASK_PATH`에 실제 파일 경로를 넣는다. mask는 binary 또는 grayscale mask 모두 가능하다.

In [ ]:
USER_IMAGE_PATH = ""  # 예: r"C:/path/to/raw.png"
USER_MASK_PATH = ""   # 예: r"C:/path/to/mask.png"

if USER_IMAGE_PATH and USER_MASK_PATH:
    image = load_image(USER_IMAGE_PATH)
    mask = load_mask(USER_MASK_PATH)
    sample_name = Path(USER_IMAGE_PATH).stem
else:
    manifest_path = DATA_ROOT / "metadata" / "samples.csv"
    if not manifest_path.exists():
        manifest = generate_random_scratch_dataset(n_samples=120, size=640, seed=7, overwrite=True)
    else:
        manifest = pd.read_csv(manifest_path)
    sample = manifest.sample(1, random_state=23).iloc[0]
    image = load_image(sample["image_path"])
    mask = load_mask(sample["mask_path"])
    sample_name = sample["sample_id"]

print(sample_name, image.shape, mask.shape, int(mask.sum()))

## 2. Scenario 정의 확인

각 scenario는 미세스크래치 제거 후보를 찾는 가설이다. 모든 scenario는 애매하면 keep하는 방향으로 설계되어 있다.

In [ ]:
scenario_summary = pd.DataFrame([
    {
        "scenario": name,
        "mode": cfg.get("mode", "all"),
        "description": cfg["description"],
        "thresholds": cfg["thresholds"],
        "hard_keep": cfg.get("hard_keep", {}),
    }
    for name, cfg in MICRO_FILTER_SCENARIOS.items()
])
display(scenario_summary)

## 3. Component feature 계산

contrast는 보조값으로만 쓰고, 주로 `area`, `width`, `elongation`, `erosion persistence`, `bbox fill`을 본다.

In [ ]:
feature_df = micro_filter_feature_table(image, mask, min_area=8)
feature_path = RUNS_ROOT / f"{sample_name}_micro_filter_features.csv"
feature_df.to_csv(feature_path, index=False, encoding="utf-8-sig")
print(feature_path)
display(feature_df[[
    "component_id", "area_px", "width_px", "major_length_px", "elongation",
    "bbox_fill_ratio", "erode_ratio_r1", "erode_ratio_r2",
    "background_excess_threshold", "contrast_z_excess"
]].head(20))

## 4. 5개 scenario 실행

In [ ]:
scenario_df, kept_masks = run_micro_filter_scenarios(image, mask, min_area=8)
scenario_path = RUNS_ROOT / f"{sample_name}_micro_filter_scenario_results.csv"
scenario_df.to_csv(scenario_path, index=False, encoding="utf-8-sig")
print(scenario_path)

summary = (
    scenario_df.groupby("scenario")
    .agg(
        components=("component_id", "count"),
        removed=("remove_micro_candidate", "sum"),
        hard_keep=("hard_keep", "sum"),
        mean_votes=("micro_votes", "mean"),
        median_width=("width_px", "median"),
        median_area=("area_px", "median"),
    )
    .reset_index()
)
display(summary)
display(scenario_df[[
    "scenario", "component_id", "remove_micro_candidate", "micro_votes", "hard_keep",
    "area_px", "width_px", "elongation", "erode_ratio_r1", "bbox_fill_ratio",
    "background_excess_threshold", "contrast_z_excess"
]].sort_values(["scenario", "remove_micro_candidate", "component_id"], ascending=[True, False, True]).head(80))

## 5. Scenario별 제거 결과 시각화

초록은 keep, 빨강은 제거 후보이다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()
axes[0].imshow(image)
axes[0].imshow(mask, cmap="Reds", alpha=0.35)
axes[0].set_title("original mask")
axes[0].axis("off")

for ax, (scenario_name, kept_mask) in zip(axes[1:], kept_masks.items()):
    overlay = overlay_removed_components(image, mask, kept_mask)
    removed_px = int(mask.sum() - kept_mask.sum())
    ax.imshow(overlay)
    ax.set_title(f"{scenario_name}\nremoved_px={removed_px}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
out_path = RUNS_ROOT / f"{sample_name}_micro_filter_scenario_overlay.png"
plt.savefig(out_path, dpi=150)
print(out_path)
plt.show()

## 6. 조건별 실패/통과 분석

실제 raw data에서 원하는 동작이 안 나오면, 어떤 조건 때문에 제거/보존됐는지 여기서 확인한다.

In [ ]:
condition_cols = [c for c in scenario_df.columns if c.startswith("cond_")]
condition_summary = scenario_df.groupby("scenario")[condition_cols].mean().reset_index()
display(condition_summary)

removed = scenario_df[scenario_df["remove_micro_candidate"]]
if len(removed) == 0:
    print("No components removed by any scenario.")
else:
    display(removed[["scenario", "component_id", "micro_votes", "area_px", "width_px", "elongation", "erode_ratio_r1", "background_excess_threshold"] + condition_cols].head(80))